In [1]:
from sqlalchemy import text
from app.db.session import session_scope
with session_scope() as s:
    v = s.execute(text("SELECT extversion FROM pg_extension WHERE extname='vector'")).scalar()
    if v:
        print(f'vector 확장 : 이미 켜져 있습니다 — {v}')
    else:
        s.execute(text('CREATE EXTENSION IF NOT EXISTS vector'))
        s.commit()
        v = s.execute(text("SELECT extversion FROM pg_extension WHERE extname='vector'")).scalar()
        print(f'vector 확장 : 켰습니다 — {v}')


vector 확장 : 이미 켜져 있습니다 — 0.8.6


In [2]:
import os
import pathlib
import sys

here = pathlib.Path.cwd()
ROOT = here.parents[2] if here.name == "day02" else here
os.chdir(ROOT)
SANDBOX = ROOT / "sandbox" / "w6" / "day02"

print("프로젝트 루트  :", ROOT)

if str(ROOT / 'backend') not in sys.path:
    sys.path.insert(0, str(ROOT / 'backend'))

SAMPLES = ROOT / "sandbox" / "w5" / "day01" / "samples"
FORMATS = SAMPLES / "_formats"

프로젝트 루트  : c:\hw\hw\python_basic\hanwha-agent


In [3]:
from app.models.document import Chunk
from app.db.session import session_scope
from app.models.document import Chunk
from sqlalchemy import func, select

queries = ['숙박', '숙박비', '출장 갈 때 숙박비 얼마?', '출장 숙박비']
zeros = []
with session_scope() as s:
    for q in queries:
        n = s.scalar(select(func.count()).select_from(Chunk).where(Chunk.text.like(f'%{q}%'))) or 0
        print(f'{q!r:<28} → {n}줄')
        if n == 0:
            zeros.append(q)
print()
print(f'넷 중 {len(zeros)}개가 0 줄입니다.')


'숙박'                         → 5줄
'숙박비'                        → 5줄
'출장 갈 때 숙박비 얼마?'             → 0줄
'출장 숙박비'                     → 0줄

넷 중 2개가 0 줄입니다.


In [4]:
import importlib
from pathlib import Path
import app.models.base
importlib.reload(app.models.base)

from app.core.config import get_settings
get_settings.cache_clear()
settings = get_settings()

print('models/base.py 의 EMBED_DIM :', app.models.base.EMBED_DIM)
print('settings.embed_dim          :', settings.embed_dim)
print('settings.embed_provider     :', settings.embed_provider)
print('settings.embed_model_dir    :', settings.embed_model_dir)
print('settings.upstage_embed_model:', settings.upstage_embed_model)
print('.env :', f'{len(Path('.env').read_text(encoding='utf-8').splitlines())}줄')


models/base.py 의 EMBED_DIM : 1024
settings.embed_dim          : 1024
settings.embed_provider     : local
settings.embed_model_dir    : C:/hw/hw/python_basic/models/bge-m3
settings.upstage_embed_model: 
.env : 68줄


In [6]:
import importlib
import inspect
from app.services import document_service

importlib.reload(document_service)

job_src = inspect.getsource(document_service.run_ingest_job)
for needle in ['range(5)', 'steps[4]["time"]', 'int(5 / len(steps)']:
    print(f'{needle:<20}: ' + ('있음' if needle in job_src else '없음'))
path = next(SAMPLES.glob('DOC-FI-009_*_v1.4.docx')).as_posix()

job_id = document_service.start_ingest_job(doc_id='DOC-FI-009', version='v2.1', path=path)   # DB에 저장된 문서 버전 확인 
document_service.run_ingest_job(job_id)
job = document_service.get_job(job_id)
print()

print(f'status={job['status']} progress={job['progress']} chunk_count={job['chunk_count']}')
if job['status'] == '실패':
    print('message:', job['message'])
for step in job['steps']:
    print(f'  {step['state']:<5} {step['name']}')

range(5)            : 있음
steps[4]["time"]    : 없음
int(5 / len(steps)  : 있음
15:41:33 INFO     sentence_transformers.base.model: No device provided, using cpu
15:41:33 INFO     sentence_transformers.base.model: Loading SentenceTransformer model from C:\hw\hw\python_basic\models\bge-m3.


Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

15:41:35 INFO     app.integrations.local_embed: 로컬 임베딩 모델 준비: C:\hw\hw\python_basic\models\bge-m3 / 1024차원


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

15:41:44 INFO     app.rag.store: 청크 적재 : DOC-FI-009 v2.1 - 19청크 - 19벡터

status=완료 progress=83 chunk_count=19
  ok    파일 검증
  ok    문서 파싱
  ok    표 -> 마크다운 변환
  ok    청킹
  ok    임베딩
  todo  검색 반영 및 현행 버전 지정


In [7]:
from sqlalchemy import select
from app.db.session import session_scope
from app.models.document import Chunk
with session_scope() as s:
    base = s.execute(select(Chunk).where(Chunk.embedding.is_not(None)).order_by(Chunk.id).limit(1)).scalar_one()
    base_vec = list(base.embedding)
    print(f'기준 청크 : {base.locator}   (벡터 길이 {len(base_vec)})')
    OPS = {'<=>  코사인 거리': Chunk.embedding.cosine_distance, '<->  L2 거리': Chunk.embedding.l2_distance, '<#>  내적의 음수': Chunk.embedding.max_inner_product}
    for name, op in OPS.items():
        print(f'\n{name}')
        stmt = select(Chunk.locator, op(base_vec).label('d')).where(Chunk.embedding.is_not(None)).order_by('d').limit(5)
        for rank, (loc, d) in enumerate(s.execute(stmt), 1):
            print(f'  {rank}. {loc:<26} {round(float(d), 4)}')


기준 청크 : DOC-FI-009_법인카드_사용_지침_v1.4   (벡터 길이 1024)

<=>  코사인 거리
  1. DOC-FI-009_법인카드_사용_지침_v1.4 0.0
  2. 제1장 제3조(발급)                0.3074
  3. 제2장 제4조(사용 범위) · 별표1       0.3164
  4. 제1장 제2조(적용 범위)             0.3389
  5. 제1장 제1조(목적)                0.3454

<->  L2 거리
  1. DOC-FI-009_법인카드_사용_지침_v1.4 0.0
  2. 제1장 제3조(발급)                0.7841
  3. 제2장 제4조(사용 범위) · 별표1       0.7954
  4. 제1장 제2조(적용 범위)             0.8233
  5. 제1장 제1조(목적)                0.8311

<#>  내적의 음수
  1. DOC-FI-009_법인카드_사용_지침_v1.4 -1.0
  2. 제1장 제3조(발급)                -0.6926
  3. 제2장 제4조(사용 범위) · 별표1       -0.6836
  4. 제1장 제2조(적용 범위)             -0.6611
  5. 제1장 제1조(목적)                -0.6546


In [8]:
with session_scope() as s:
    stmt = select(Chunk.locator, Chunk.embedding.cosine_distance(base_vec).label('d')).where(Chunk.embedding.is_not(None)).order_by('d').limit(5)
    rows = [(loc, round(float(d), 4), round(1.0 - float(d), 4)) for loc, d in s.execute(stmt)]
by_distance = rows
by_score = sorted(rows, key=lambda r: r[2], reverse=True)
print('[ distance 오름차순 ]        [ score 내림차순 ]')
for a, b in zip(by_distance, by_score):
    print(f'  {a[1]:<8}{a[0]:<22} {b[2]:<8}{b[0]}')
print('\n두 표의 순서가 같은가 :', [r[0] for r in by_distance] == [r[0] for r in by_score])


[ distance 오름차순 ]        [ score 내림차순 ]
  0.0     DOC-FI-009_법인카드_사용_지침_v1.4 1.0     DOC-FI-009_법인카드_사용_지침_v1.4
  0.3074  제1장 제3조(발급)            0.6926  제1장 제3조(발급)
  0.3164  제2장 제4조(사용 범위) · 별표1   0.6836  제2장 제4조(사용 범위) · 별표1
  0.3389  제1장 제2조(적용 범위)         0.6611  제1장 제2조(적용 범위)
  0.3454  제1장 제1조(목적)            0.6546  제1장 제1조(목적)

두 표의 순서가 같은가 : True


In [9]:
from app.rag import embedder, store

QUESTION = "출장 갈 때 숙박비 얼마?"

vec = embedder.embed_query(QUESTION)
print("질문 : ", QUESTION)
print("질문 벡터 : ", len(vec))

with session_scope() as s:
    rows = store.vector_search(s, vec, limit=12)
print() 

for i, r in enumerate(rows, 1):
    print(f"{i}. {r['doc_id']} {r['version']} - {r['locator']}   score {r['score']}")
    print("  ", r['text'][:40].replace("\n", " "), "...")

print(f"벡터로 찾은 건수 : {len(rows)}건")

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

질문 :  출장 갈 때 숙박비 얼마?
질문 벡터 :  1024

1. DOC-FI-009 v2.1 - 제2장 제11조(출장 중 사용)   score 0.557
   제11조(출장 중 사용) ① 출장 중 지출은 법인카드 사용을 원칙으로 한 ...
2. DOC-FI-009 v2.1 - 표2   score 0.5371
   | 구분 | 항목 | 사용 가능 | 증빙 추가 사항 | |---|---| ...
3. DOC-FI-009 v2.1 - 표1   score 0.4539
   | 버전 | 구분 | 시행일 | 주요 내용 | 작성 | 검토 | 승인 | ...
4. DOC-FI-009 v2.1 - 제2장 제9조(사용 한도)   score 0.4523
   제9조(사용 한도) ① 1회 결제 한도는 1,000,000원으로 한다.  ...
5. DOC-FI-009 v2.1 - 제3장 제12조(증빙 제출)   score 0.4202
   제12조(증빙 제출) ① 사용일이 속한 달의 다음 달 5일까지 증빙을 제 ...
6. DOC-FI-009 v2.1 - 제2장 제7조(사용 제한)   score 0.4191
   제7조(사용 제한) ① 심야(23시 이후) 및 휴일 사용은 사전 승인을  ...
7. DOC-FI-009 v2.1 - DOC-FI-009_법인카드_사용_지침_v1.4   score 0.3967
   이 표는 문서관리규정 제9조에 따라 관리한다. 목차 제1조(목적) 제2조 ...
8. DOC-FI-009 v2.1 - 제3장 제13조(미제출)   score 0.3871
   제13조(미제출) ① 증빙을 제출하지 아니한 경우 인사총무팀은 1회 이상 ...
9. DOC-FI-009 v2.1 - 제2장 제4조(사용 범위) · 별표1   score 0.3686
   제4조(사용 범위) ① 법인카드는 업무 수행에 직접 필요한 경비의 결제에 ...
10. DOC-FI-009 v2.1 - 제3장 제15조(부정 사용)   score 0.3555
   제15조(부정 사용) ① 개인적 용도로